# From Prototype to Production Workflow

A **prototype** multi-agent workflow proves the idea: agents collaborate, a draft emerges, stakeholders say "this could work." **Production** means the same workflow survives real traffic — with guardrails, observability, contracts, evals, and safe side effects.

The gap is not "swap demo data for a database." It is **operational maturity**: every run is traced, bounded, validated, and recoverable.

This notebook walks **ReleaseFlow** from a minimal prototype to a production-shaped orchestrator, adding one hardening layer at a time:

1. Structured tracing
2. Guardrails (timeouts, max steps, cost caps)
3. Agent contracts and validation
4. Idempotent side effects
5. Human-in-the-loop gates
6. Eval harness and readiness scoring

Everything is self-contained plain Python. No frameworks or prior notebooks are required.

In [ ]:
"""
ReleaseFlow — prototype → production hardening lab.
"""

import json
import os
import time
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


CHANGELOG: list[dict[str, str]] = [
    {"version": "2.4.0", "item": "Added bulk export endpoint for reports"},
    {"version": "2.4.0", "item": "Fixed session timeout on mobile browsers"},
    {"version": "2.4.0", "item": "Deprecated legacy v1 webhook format"},
]

STYLE_GUIDE = {"max_words": 200}
PUBLISHED: list[dict[str, Any]] = []
IDEMPOTENCY_REGISTRY: dict[str, str] = {}

print(f"ReleaseFlow corpus: {len(CHANGELOG)} changelog items")

---

## 1. Prototype vs Production

| Dimension | Prototype | Production |
|-----------|-----------|------------|
| **Goal** | Prove feasibility | Reliable at scale |
| **Tracing** | `print()` | Structured JSON traces |
| **Limits** | None | max_steps, timeouts, token budget |
| **Contracts** | Ad-hoc dict keys | Validated reads/writes |
| **Side effects** | Direct writes | Idempotency keys |
| **Humans** | Skipped | Gates before publish |
| **Quality** | Eyeball demo | Eval harness + scorecard |
| **Config** | Hard-coded | External env / config |
| **Versioning** | "latest" | Pinned graph + prompt version |

---

## 2. Maturity Model

| Level | Name | Characteristics |
|-------|------|-----------------|
| **L0** | Sketch | Single script, no trace |
| **L1** | Prototype | Multi-agent, happy path only |
| **L2** | Hardened | Traces, caps, contracts |
| **L3** | Production | HITL, idempotency, evals |
| **L4** | Operated | Monitoring, versioning, rollback |

This notebook takes ReleaseFlow from **L1 → L3** in code you can extend to L4.

---

## 3. Shared Workflow State

In [ ]:
@dataclass
class WorkflowState:
    run_id: str
    goal: str
    artifacts: dict[str, Any] = field(default_factory=dict)
    status: str = "running"
    step_count: int = 0
    trace: list[dict[str, Any]] = field(default_factory=list)
    errors: list[str] = field(default_factory=list)
    metadata: dict[str, Any] = field(default_factory=dict)

    def log_step(self, agent: str, detail: str, **extra: Any) -> None:
        self.step_count += 1
        self.trace.append({
            "step": self.step_count,
            "agent": agent,
            "detail": detail,
            "ts": datetime.now(timezone.utc).isoformat(),
            **extra,
        })


AGENT_CONTRACTS: dict[str, dict[str, list[str]]] = {
    "researcher": {"reads": ["goal"], "writes": ["research"]},
    "writer": {"reads": ["research"], "writes": ["draft"]},
    "critic": {"reads": ["draft"], "writes": ["approved", "review_feedback"]},
    "publisher": {"reads": ["draft", "approved"], "writes": ["announcement_id", "human_ok"]},
}


def validate_contract(agent: str, state: WorkflowState) -> list[str]:
    contract = AGENT_CONTRACTS.get(agent, {})
    missing = []
    for key in contract.get("reads", []):
        if key == "goal":
            continue
        if key == "review_feedback":
            continue
        if key not in state.artifacts:
            missing.append(key)
    return missing


print("WorkflowState and contracts ready")

---

## 4. L1 — Prototype Workflow (Happy Path Only)

Fast to build. No guardrails. Fine for demos — dangerous in production.

In [ ]:
class PrototypeWorkflow:
    """L1 — minimal sequential agents, no validation or limits."""

    name = "prototype"
    maturity = "L1"

    def researcher(self, state: WorkflowState) -> None:
        state.artifacts["research"] = {
            "version": "2.4.0",
            "changes": [c["item"] for c in CHANGELOG],
        }

    def writer(self, state: WorkflowState) -> None:
        facts = state.artifacts["research"]
        state.artifacts["draft"] = (
            f"Release v{facts['version']} is now available.\n\n"
            f"What's new: " + "; ".join(facts["changes"]) + ".\n\n"
            f"Breaking changes: Legacy v1 webhook deprecated.\n\n"
            f"Questions? Contact #releases."
        )

    def critic(self, state: WorkflowState) -> None:
        draft = state.artifacts["draft"]
        state.artifacts["approved"] = "v2.4" in draft and "#releases" in draft

    def publisher(self, state: WorkflowState) -> None:
        ann = {"id": f"ANN-{uuid.uuid4().hex[:6].upper()}", "body": state.artifacts["draft"]}
        PUBLISHED.append(ann)
        state.artifacts["announcement_id"] = ann["id"]
        state.status = "published"

    def run(self, goal: str) -> WorkflowState:
        state = WorkflowState(run_id=f"proto_{uuid.uuid4().hex[:6]}", goal=goal)
        for agent, fn in [
            ("researcher", self.researcher),
            ("writer", self.writer),
            ("critic", self.critic),
            ("publisher", self.publisher),
        ]:
            fn(state)
        return state


proto = PrototypeWorkflow().run("v2.4.0 announcement")
print(f"Prototype: status={proto.status} steps={proto.step_count} trace_events={len(proto.trace)}")
print(f"Announcement: {proto.artifacts.get('announcement_id')}")

---

## 5. Gap Analysis — What the Prototype Lacks

In [ ]:
PRODUCTION_GAPS = [
    ("tracing", "No structured trace — cannot debug failed runs"),
    ("guardrails", "No max_steps — review loops can run forever"),
    ("contracts", "No validation — writer can run before research"),
    ("idempotency", "Retry duplicates announcements"),
    ("human_gate", "Publishes without human approval"),
    ("evals", "No regression suite for prompt/graph changes"),
    ("config", "Limits and flags hard-coded"),
    ("versioning", "No workflow version pinned per run"),
]

PROTOTYPE_HAS: set[str] = set()  # L1 prototype implements none of the production layers

print(f"{'Gap':<14} Issue")
print("-" * 60)
for gap, issue in PRODUCTION_GAPS:
    status = "MISSING" if gap not in PROTOTYPE_HAS else "ok"
    print(f"{gap:<14} {issue} [{status}]")

---

## 6. Layer 1 — Structured Tracing

In [ ]:
class TracedAgentRunner:
    """Wraps agent steps with timing and JSON trace events."""

    def run_step(self, state: WorkflowState, agent: str, fn: Callable[[WorkflowState], None]) -> None:
        t0 = time.perf_counter()
        try:
            fn(state)
            ms = round((time.perf_counter() - t0) * 1000, 2)
            state.log_step(agent, "ok", ms=ms, status="ok")
        except Exception as exc:
            ms = round((time.perf_counter() - t0) * 1000, 2)
            state.log_step(agent, str(exc), ms=ms, status="error")
            state.errors.append(f"{agent}: {exc}")
            raise

    def trace_jsonl(self, state: WorkflowState) -> str:
        return "\n".join(json.dumps({**e, "run_id": state.run_id}) for e in state.trace)


runner = TracedAgentRunner()
demo_state = WorkflowState(run_id="trace_demo", goal="test")
runner.run_step(demo_state, "researcher", lambda s: s.artifacts.update({"research": {}}))
print(runner.trace_jsonl(demo_state))

---

## 7. Layer 2 — Guardrails

| Guardrail | Purpose |
|-----------|--------|
| `max_steps` | Cap total agent invocations |
| `max_revisions` | Cap write↔review loops |
| `step_timeout_ms` | Fail slow agents |
| `token_budget` | Stop before context explosion |

In [ ]:
@dataclass
class WorkflowGuardrails:
    max_steps: int = 12
    max_revisions: int = 2
    step_timeout_ms: float = 5000.0
    token_budget: int = 8000

    def check_step_limit(self, state: WorkflowState) -> bool:
        return state.step_count < self.max_steps

    def check_revision_limit(self, revision_count: int) -> bool:
        return revision_count <= self.max_revisions


DEFAULT_GUARDRAILS = WorkflowGuardrails()
print(f"Guardrails: max_steps={DEFAULT_GUARDRAILS.max_steps}, max_revisions={DEFAULT_GUARDRAILS.max_revisions}")

---

## 8. Layer 3 — Contract Validation Before Each Step

In [ ]:
class ContractEnforcer:
    def enforce(self, agent: str, state: WorkflowState) -> None:
        missing = validate_contract(agent, state)
        if missing:
            raise ValueError(f"contract violation before {agent}: missing {missing}")


enforcer = ContractEnforcer()
bad_state = WorkflowState(run_id="bad", goal="test")
try:
    enforcer.enforce("writer", bad_state)
except ValueError as exc:
    print(f"Caught: {exc}")

---

## 9. Layer 4 — Idempotent Publisher

Production publish steps must survive **retries** without duplicate side effects.

In [ ]:
def publish_idempotent(state: WorkflowState) -> str:
    """Publish announcement once per run_id."""
    if state.run_id in IDEMPOTENCY_REGISTRY:
        existing = IDEMPOTENCY_REGISTRY[state.run_id]
        state.log_step("publisher", f"idempotent skip → {existing}", status="skipped")
        return existing
    ann_id = f"ANN-{uuid.uuid4().hex[:6].upper()}"
    PUBLISHED.append({"id": ann_id, "body": state.artifacts.get("draft", ""), "run_id": state.run_id})
    IDEMPOTENCY_REGISTRY[state.run_id] = ann_id
    state.artifacts["announcement_id"] = ann_id
    return ann_id


idem_state = WorkflowState(run_id="run_idem_1", goal="test")
idem_state.artifacts["draft"] = "Release v2.4.0..."
a = publish_idempotent(idem_state)
b = publish_idempotent(idem_state)
print(f"First: {a} | Retry: {b} | Same: {a == b}")

---

## 10. Layer 5 — Human-in-the-Loop Gate

In [ ]:
@dataclass
class HumanGateConfig:
    require_approval: bool = True
    auto_approve_in_dev: bool = False


def human_gate(state: WorkflowState, config: HumanGateConfig) -> bool:
    if not state.artifacts.get("approved"):
        state.log_step("human_gate", "blocked — not approved", status="blocked")
        return False
    if config.require_approval and not config.auto_approve_in_dev:
        state.log_step("human_gate", "awaiting human click", status="pending")
        return False
    state.artifacts["human_ok"] = True
    state.log_step("human_gate", "approved", status="ok")
    return True


gate_state = WorkflowState(run_id="gate", goal="test")
gate_state.artifacts["approved"] = True
print("Dev auto-approve:", human_gate(gate_state, HumanGateConfig(auto_approve_in_dev=True)))
prod_gate_state = WorkflowState(run_id="g2", goal="test")
prod_gate_state.artifacts["approved"] = True
print("Prod pending:", human_gate(prod_gate_state, HumanGateConfig(require_approval=True, auto_approve_in_dev=False)))

---

## 11. Layer 6 — Eval Harness

Regression tests for workflow outcomes — run on every graph or prompt change.

In [ ]:
@dataclass
class EvalCase:
    name: str
    goal: str
    expect_approved: bool
    expect_published: bool
    max_steps: int = 15


EVAL_SUITE: list[EvalCase] = [
    EvalCase("happy_path", "v2.4.0 customer announcement", True, True),
    EvalCase("missing_goal_data", "announcement without version", True, True),
]


def run_eval(workflow: Any, cases: list[EvalCase]) -> list[dict[str, Any]]:
    results = []
    for case in cases:
        state = workflow.run(case.goal)
        approved = bool(state.artifacts.get("approved"))
        published = state.status == "published"
        ok = (approved == case.expect_approved and published == case.expect_published
              and state.step_count <= case.max_steps)
        results.append({
            "case": case.name,
            "pass": ok,
            "approved": approved,
            "published": published,
            "steps": state.step_count,
        })
    return results


print("Eval harness ready — wired to ProductionWorkflow in section 12")

---

## 12. L3 — Production Workflow (All Layers Combined)

In [ ]:
@dataclass
class ProductionConfig:
    workflow_version: str = "2.0.0"
    guardrails: WorkflowGuardrails = field(default_factory=WorkflowGuardrails)
    human_gate: HumanGateConfig = field(default_factory=lambda: HumanGateConfig(auto_approve_in_dev=True))
    environment: str = "development"


class ProductionWorkflow:
    """L3 — traced, guarded, contracted, idempotent, gated release pipeline."""

    name = "production"
    maturity = "L3"

    def __init__(self, config: ProductionConfig | None = None) -> None:
        self.config = config or ProductionConfig()
        self.runner = TracedAgentRunner()
        self.enforcer = ContractEnforcer()

    def _researcher(self, state: WorkflowState) -> None:
        state.artifacts["research"] = {"version": "2.4.0", "changes": [c["item"] for c in CHANGELOG]}

    def _writer(self, state: WorkflowState) -> None:
        facts = state.artifacts["research"]
        feedback = state.artifacts.get("review_feedback", "")
        draft = (
            f"Release v{facts['version']} is now available.\n\n"
            f"What's new: " + "; ".join(facts["changes"]) + ".\n\n"
            f"Breaking changes: Legacy v1 webhook format is deprecated.\n\n"
            f"Questions? Contact #releases."
        )
        if feedback and "too long" in feedback:
            draft = "\n".join(draft.split("\n")[:4]) + "\n\nQuestions? Contact #releases."
        state.artifacts["draft"] = draft

    def _critic(self, state: WorkflowState) -> None:
        draft = state.artifacts.get("draft", "")
        issues = []
        if "v2.4" not in draft:
            issues.append("missing version")
        if "Breaking changes" not in draft:
            issues.append("missing breaking section")
        if "#releases" not in draft:
            issues.append("missing support channel")
        if len(draft.split()) > STYLE_GUIDE["max_words"]:
            issues.append(f"too long ({len(draft.split())} words)")
        if issues:
            state.artifacts["approved"] = False
            state.artifacts["review_feedback"] = ", ".join(issues)
        else:
            state.artifacts["approved"] = True
            state.artifacts["review_feedback"] = ""

    def _publisher(self, state: WorkflowState) -> None:
        if not human_gate(state, self.config.human_gate):
            state.status = "pending_human"
            return
        publish_idempotent(state)
        state.status = "published"

    def _run_agent(self, state: WorkflowState, agent: str, fn: Callable[[WorkflowState], None]) -> bool:
        if not self.config.guardrails.check_step_limit(state):
            state.status = "failed"
            state.errors.append("max_steps exceeded")
            state.log_step("guardrail", "max_steps exceeded", status="error")
            return False
        self.enforcer.enforce(agent, state)
        self.runner.run_step(state, agent, fn)
        return True

    def run(self, goal: str, *, force_long_draft: bool = False) -> WorkflowState:
        state = WorkflowState(
            run_id=f"prod_{uuid.uuid4().hex[:8]}",
            goal=goal,
            metadata={"workflow_version": self.config.workflow_version, "env": self.config.environment},
        )
        if not self._run_agent(state, "researcher", self._researcher):
            return state

        revisions = 0
        while True:
            if not self._run_agent(state, "writer", self._writer):
                return state
            if force_long_draft and revisions == 0:
                state.artifacts["draft"] = state.artifacts.get("draft", "") + " word" * 80
            if not self._run_agent(state, "critic", self._critic):
                return state
            if state.artifacts.get("approved"):
                break
            revisions += 1
            if not self.config.guardrails.check_revision_limit(revisions):
                state.status = "failed"
                state.errors.append("max_revisions exceeded")
                return state

        if not self._run_agent(state, "publisher", self._publisher):
            return state
        return state


prod = ProductionWorkflow()
prod_result = prod.run("v2.4.0 production announcement")
print(f"Production: status={prod_result.status} steps={prod_result.step_count}")
print(f"Trace events: {len(prod_result.trace)}")
print(f"Version: {prod_result.metadata.get('workflow_version')}")
print(prod.runner.trace_jsonl(prod_result))

---

## 13. Production Readiness Scorecard

In [ ]:
READINESS_CRITERIA = [
    ("structured_trace", lambda s: len(s.trace) > 0),
    ("workflow_version", lambda s: "workflow_version" in s.metadata),
    ("guardrails_active", lambda s: True),
    ("contracts_enforced", lambda s: not any("contract" in e for e in s.errors)),
    ("idempotent_publish", lambda s: s.run_id in IDEMPOTENCY_REGISTRY or s.status != "published"),
    ("human_gate_checked", lambda s: "human_ok" in s.artifacts or s.status == "pending_human"),
    ("no_unhandled_errors", lambda s: not s.errors),
    ("published_or_explicit_fail", lambda s: s.status in ("published", "failed", "pending_human")),
]


def readiness_score(state: WorkflowState) -> dict[str, Any]:
    checks = {name: fn(state) for name, fn in READINESS_CRITERIA}
    passed = sum(checks.values())
    return {"score": f"{passed}/{len(checks)}", "checks": checks, "ready": passed == len(checks)}


proto_score = readiness_score(proto)
prod_score = readiness_score(prod_result)
print(f"Prototype readiness: {proto_score['score']} ready={proto_score['ready']}")
print(f"Production readiness: {prod_score['score']} ready={prod_score['ready']}")
print("\nProduction checks:")
for k, v in prod_score["checks"].items():
    print(f"  {'✓' if v else '✗'} {k}")

---

## 14. External Configuration

Production workflows read limits from **environment** or config service — not literals in code.

In [ ]:
def load_config_from_env() -> ProductionConfig:
    return ProductionConfig(
        workflow_version=os.environ.get("RELEASEFLOW_VERSION", "2.0.0"),
        environment=os.environ.get("RELEASEFLOW_ENV", "development"),
        guardrails=WorkflowGuardrails(
            max_steps=int(os.environ.get("RELEASEFLOW_MAX_STEPS", "12")),
            max_revisions=int(os.environ.get("RELEASEFLOW_MAX_REVISIONS", "2")),
        ),
        human_gate=HumanGateConfig(
            require_approval=os.environ.get("RELEASEFLOW_REQUIRE_HUMAN", "false").lower() == "true",
            auto_approve_in_dev=os.environ.get("RELEASEFLOW_ENV", "development") != "production",
        ),
    )


cfg = load_config_from_env()
print(pretty({
    "version": cfg.workflow_version,
    "env": cfg.environment,
    "max_steps": cfg.guardrails.max_steps,
    "require_human": cfg.human_gate.require_approval,
}))

---

## 15. Run Eval Suite

In [ ]:
eval_results = run_eval(ProductionWorkflow(), EVAL_SUITE)
print(f"{'Case':<25} {'Pass':<6} Steps")
print("-" * 40)
for r in eval_results:
    print(f"{r['case']:<25} {str(r['pass']):<6} {r['steps']}")
print(f"\nPass rate: {sum(r['pass'] for r in eval_results)}/{len(eval_results)}")

---

## 16. L4 — Operations (Monitoring Metrics)

At **L4**, export aggregates from traces to dashboards:

In [ ]:
@dataclass
class WorkflowMetrics:
    run_id: str
    status: str
    total_steps: int
    total_ms: float
    error_count: int
    published: bool


def extract_metrics(state: WorkflowState) -> WorkflowMetrics:
    total_ms = sum(e.get("ms", 0) for e in state.trace)
    return WorkflowMetrics(
        run_id=state.run_id,
        status=state.status,
        total_steps=state.step_count,
        total_ms=total_ms,
        error_count=len(state.errors),
        published=state.status == "published",
    )


metrics = extract_metrics(prod_result)
print(pretty(metrics))

MONITORING_ALERTS = [
    ("error_rate", "errors / total_runs > 5%"),
    ("p95_latency", "total_ms p95 > SLA"),
    ("human_pending", "status=pending_human count > threshold"),
    ("eval_regression", "eval pass rate drops vs baseline"),
]
print("\nSuggested alerts:")
for name, rule in MONITORING_ALERTS:
    print(f"  - {name}: {rule}")

---

## 17. Migration Checklist — Prototype → Production

In [ ]:
MIGRATION_CHECKLIST = [
    "Add structured trace per agent step",
    "Set max_steps and max_revisions",
    "Define and enforce agent contracts",
    "Idempotency keys on all write side effects",
    "Human gate before irreversible actions",
    "Externalize config (env / config service)",
    "Pin workflow_version on every run",
    "Build eval suite — block deploy on regression",
    "Wire metrics to monitoring",
    "Document rollback procedure for graph/prompt versions",
]

print("Prototype → Production migration:")
for i, item in enumerate(MIGRATION_CHECKLIST, 1):
    print(f"  {i:2}. [ ] {item}")

---

## 18. Side-by-Side Comparison

In [ ]:
COMPARISON = [
    ("Maturity", PrototypeWorkflow.maturity, ProductionWorkflow.maturity),
    ("Trace events", len(proto.trace), len(prod_result.trace)),
    ("Step count tracked", proto.step_count > 0, prod_result.step_count > 0),
    ("Workflow version", "none", prod_result.metadata.get("workflow_version")),
    ("Readiness", proto_score["score"], prod_score["score"]),
]

print(f"{'Dimension':<22} {'Prototype':<18} Production")
print("-" * 55)
for dim, p, pr in COMPARISON:
    print(f"{dim:<22} {str(p):<18} {pr}")

---

## 19. Optional Live LLM Writer

In [ ]:
USE_LIVE_LLM = False

if USE_LIVE_LLM:
    class LiveProductionWorkflow(ProductionWorkflow):
        def _writer(self, state: WorkflowState) -> None:
            from openai import OpenAI
            facts = state.artifacts["research"]
            client = OpenAI(api_key=OPENAI_API_KEY)
            resp = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": f"Write release announcement: {facts}"}],
                temperature=0.3,
            )
            state.artifacts["draft"] = resp.choices[0].message.content or ""
    result = LiveProductionWorkflow().run("Live production demo")
    print(result.status)
else:
    print("Offline mode — production hardening layers work without an LLM.")
    print("Set USE_LIVE_LLM = True to swap the writer step only.")

---

## 20. Quiz

1. Name three gaps between a prototype and production workflow.
2. Why idempotency keys on publish steps?
3. What does the readiness scorecard measure?
4. When should `pending_human` status appear?
5. What triggers an eval regression alert?

---

## 21. Summary

| Layer | Production addition |
|-------|---------------------|
| **L1 Prototype** | Happy-path agents, proves idea |
| **Tracing** | JSON events with timing |
| **Guardrails** | max_steps, max_revisions |
| **Contracts** | Validate reads before each agent |
| **Idempotency** | One publish per run_id |
| **Human gate** | Block publish until approved |
| **Evals** | Regression suite on changes |
| **L4 Operations** | Metrics, alerts, versioned rollback |

Ship prototypes to learn. Ship **production workflows** when every run is traced, bounded, validated, and recoverable — and when evals gate every deploy.